In [7]:
import pandas as pd


df = pd.read_csv('dataset_imot-bg-scraper-bulgaria-real-estate_2026-10-22_10-38-56-578.csv') 


display(df.head(10))

print("Columns count:", df.columns)
print("Rows and columns count:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_imot-bg-scraper-bulgaria-real-estate_2026-10-22_10-38-56-578.csv'

In [ ]:
import pandas as pd

# 1. Drop redundant or unnecessary columns
# We keep descriptionBg, cityBg, titleBg and drop their English/duplicate counterparts
columns_to_drop = [
    'agencyUrl', 
    'description', 
    'city', 
    'title', 
    'imageThumbnail', 
    'imageUrls', 
    'scrapedAt'
]

# Drop only the columns that actually exist in the dataframe to avoid errors
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=existing_columns_to_drop)

# 2. Inspect missing values (NaNs) per column
print("=== Missing values per column ===")
print(df_cleaned.isnull().sum())

# 3. Show the new shape of the dataframe
print("\nNew dataset shape (rows, columns):", df_cleaned.shape)

In [ ]:
# 1. Drop completely empty columns like publishedDate
if 'publishedDate' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['publishedDate'])

# 2. Check rows where critical data like price or area might be missing
print("Missing prices:", df_cleaned['price'].isnull().sum())
print("Missing areas:", df_cleaned['area'].isnull().sum())

# 3. Show current columns and data types
print("\n=== Cleaned Dataset Columns & Types ===")
print(df_cleaned.dtypes)

In [ ]:
# 1. Drop all image URL columns automatically by finding columns starting with 'imageUrls'
image_columns = [col for col in df_cleaned.columns if col.startswith('imageUrls')]
df_cleaned = df_cleaned.drop(columns=image_columns)

# 2. Check unique values for categorical features to understand our dataset scope
print("=== Listing Types ===")
print(df_cleaned['listingType'].value_counts())

print("\n=== Property Types ===")
print(df_cleaned['propertyType'].value_counts())

# 3. Show final shape after dropping images
print("\nDataset shape after cleaning images:", df_cleaned.shape)

In [ ]:
import os

# 1. Make sure the 'processed' directory exists
os.makedirs('processed', exist_ok=True)

# 2. Save the cleaned dataframe to a new CSV file in the processed folder
output_path = 'processed/cleaned_prodazhbi_05_05_2026.csv'
df_cleaned.to_csv(output_path, index=False)

print(f"Cleaned dataset successfully saved to: {output_path}")
print(f"Final dataset has {df_cleaned.shape[0]} rows and {df_cleaned.shape[1]} columns.")


In [ ]:
import pandas as pd

# 1. Load our cleaned dataset from the processed folder
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Inspect yearBuilt and floors formatting
print("=== Before cleaning data types ===")
print(df[['yearBuilt', 'floor', 'totalFloors']].head())

# 3. Handle missing values in yearBuilt (e.g., fill with median year or a placeholder like 0 / unknown)
# For real estate, filling with median year is a common strategy, or keeping them as a special category
median_year = df['yearBuilt'].median()
df['yearBuilt'] = df['yearBuilt'].fillna(median_year).astype(int)

# 4. Handle floor and totalFloors (convert float to int safely)
df['floor'] = df['floor'].fillna(0).astype(int)
df['totalFloors'] = df['totalFloors'].fillna(0).astype(int)

print("\n=== After converting to integers ===")
print(df[['yearBuilt', 'floor', 'totalFloors']].dtypes)
print(df[['yearBuilt', 'floor', 'totalFloors']].head())

In [ ]:
import pandas as pd

# 1. Load the processed dataset
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Show basic descriptive statistics for numerical columns (price, area, pricePerSqm)
print("=== Numerical Statistics (Price, Area, Price per Sqm) ===")
print(df[['price', 'area', 'pricePerSqm']].describe())

# 3. Check construction types available in our dataset
print("\n=== Construction Types ===")
print(df['constructionType'].value_counts())

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Set Pandas display format to round floats to 2 decimal places and avoid scientific notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 3. Show statistics rounded to 2 decimal places
print("=== Formatted Statistics (Price, Area, Price per Sqm) ===")
display(df[['price', 'area', 'pricePerSqm']].describe())

In [ ]:
# 1. Filter out unrealistic outliers for area and price
# Realistic apartment area: between 15 sqm and 400 sqm
# Minimum reasonable price: e.g., above 5,000 EUR
df_filtered = df[(df['area'] >= 15) & (df['area'] <= 400) & (df['price'] > 5000)].copy()

# 2. Compare row counts
print(f"Original rows count: {len(df)}")
print(f"Filtered rows count: {len(df_filtered)}")
print(f"Removed outliers: {len(df) - len(df_filtered)} rows")

# 3. Show updated statistics after filtering
print("\n=== Statistics After Outlier Removal ===")
display(df_filtered[['price', 'area', 'pricePerSqm']].describe())

In [ ]:
# Save the final filtered and cleaned dataset
output_path = 'processed/cleaned_prodazhbi_05_05_2026.csv'
df_filtered.to_csv(output_path, index=False)

print(f"Final clean dataset successfully saved to: {output_path}")
print(f"Total rows ready for modeling: {len(df_filtered)}")

In [ ]:
import pandas as pd

# 1. Load the cleaned dataset
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Check which columns contain text (categorical data)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns found:", categorical_cols)

# 3. Apply One-Hot Encoding to convert text categories into numerical columns (0 and 1)
# drop_first=True helps avoid multicollinearity in linear models
df_encoded = pd.get_dummies(df, columns=['constructionType', 'neighborhoodBg'], drop_first=True)

# 4. Show the difference in shape
print(f"\nShape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")

# 5. Preview the first 3 rows of the newly encoded dataset
display(df_encoded.head(3))

In [ ]:
import pandas as pd

# 1. Load the cleaned dataset
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Define the target variable (what we want to predict)
y = df['price']

# 3. Select only the relevant features for our model (no raw text descriptions, no pricePerSqm)
selected_features = [
    'area', 'floor', 'totalFloors', 'yearBuilt', 
    'constructionType', 'neighborhoodBg', 'rooms', 'propertyType'
]

X = df[selected_features].copy()

# 4. Apply One-Hot Encoding to convert categorical features (text) into numbers (0 and 1)
X_encoded = pd.get_dummies(X, drop_first=True)

# 5. Check the final shapes
print(f"Features shape (X_encoded): {X_encoded.shape}")
print(f"Target shape (y): {y.shape}")

# Preview the first 5 rows of our processed features
display(X_encoded.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# 1. Handle missing values by filling them with the median of each column
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_encoded)

# 2. Split data into Training set (80%) and Testing set (20%)
# random_state ensures reproducibility (we get the same split every time we run it)
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# 3. Print the shapes to verify
print(f"Training features shape (X_train): {X_train.shape}")
print(f"Testing features shape (X_test): {X_test.shape}")
print(f"Training target shape (y_train): {y_train.shape}")
print(f"Testing target shape (y_test): {y_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Initialize the Random Forest Regressor model
# random_state ensures we get consistent results every time we run it
model = RandomForestRegressor(random_state=42)

# 2. Train the model using the training data
model.fit(X_train, y_train)

# 3. Make predictions on the test dataset
y_pred = model.predict(X_test)

# 4. Evaluate the model performance
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"=== Model Performance ===")
print(f"Mean Absolute Error (MAE): {mae:.2f} EUR")
print(f"R2 Score (Accuracy): {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load cleaned dataset
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# 2. Drop Data Leakage & Unnecessary Identifier columns (URLs, IDs, prices)
columns_to_drop = ['price']
for col in ['pricePerSqm', 'priceFormatted', 'descriptionBg', 'listingUrl']:
    if col in df.columns:
        columns_to_drop.append(col)

for col in df.columns:
    if 'url' in col.lower() or 'id' in col.lower() or 'title' in col.lower():
        if col not in columns_to_drop:
            columns_to_drop.append(col)

X = df.drop(columns=columns_to_drop)
y = df['price']

# 3. FEATURE ENGINEERING: Calculate mean price per neighborhood/city (Target Encoding)
if 'neighborhoodBg' in df.columns:
    location_col = 'neighborhoodBg'
elif 'cityBg' in df.columns:
    location_col = 'cityBg'
else:
    location_col = None

if location_col:
    location_means = df.groupby(location_col)['price'].mean()
    X['location_price_tier'] = df[location_col].map(location_means)
    X = X.drop(columns=[location_col])

# 4. One-hot encode remaining small categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# 5. Imputer & Train-Test Split
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_encoded)
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# 6. Fit Gradient Boosting Model
gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)

# 7. Evaluate
y_pred_gb = gb_model.predict(X_test)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f"=== Final Clean Model Performance ===")
print(f"Mean Absolute Error (MAE): {mae_gb:.2f} EUR")
print(f"R2 Score (Accuracy): {r2_gb:.4f}")

# 8. Feature Importance
feature_importances = pd.Series(gb_model.feature_importances_, index=X_encoded.columns)
print("\n=== Top 5 Most Important Features ===")
print(feature_importances.nlargest(5))

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# Речник с моделите, които ще тестваме
models = {
    "Linear Regression (Ridge)": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

model_results = []

for name, model in models.items():
    # Обучение
    model.fit(X_train, y_train)
    
    # Предсказание и оценка
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    model_results.append({
        "Model": name,
        "MAE (EUR)": round(mae, 2),
        "R2 Score": round(r2, 4)
    })

# Превръщаме резултатите в красива таблица
comparison_df = pd.DataFrame(model_results)
print("=== Comparing models ===")
print(comparison_df.to_string(index=False))

In [ ]:
# Добави тези думи в списъка за изключване в началото на скрипта:
columns_to_drop = ['price']
for col in ['pricePerSqm', 'priceFormatted', 'descriptionBg', 'listingUrl']:
    if col in df.columns:
        columns_to_drop.append(col)

# Динамично премахване на адреси, телефони, имена на агенции и ID-та
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['url', 'id', 'title', 'address', 'agency', 'phone', 'name']):
        if col not in columns_to_drop:
            columns_to_drop.append(col)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Извличаме само топ 3 реални фактора
feature_importances = pd.Series(gb_model.feature_importances_, index=X_encoded.columns)
top_3_features = feature_importances.nlargest(3)

# Създаваме чиста графика
plt.figure(figsize=(9, 4))
sns.barplot(
    x=top_3_features.values, 
    y=top_3_features.index, 
    hue=top_3_features.index, 
    palette='viridis', 
    legend=False
)

plt.title('Top 3 Key Price Drivers in Bulgaria Real Estate', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
def predict_property_price(area_sqm, year, neighborhood_name):
    """
    Function to predict the price of a new property using our trained Gradient Boosting model.
    """
    # 1. Create a base dictionary with the core property features
    new_property = {
        'area': area_sqm,
        'yearBuilt': year,
    }
    
    # 2. Convert to a DataFrame
    pred_df = pd.DataFrame([new_property])
    
    # 3. Calculate or map the location_price_tier for the specified neighborhood/city
    # (Here we use the pre-calculated mean price per location from our training dataset)
    if neighborhood_name in location_means:
        loc_tier = location_means[neighborhood_name]
    else:
        # Fallback to the overall training mean if the location is unknown or new
        loc_tier = y_train.mean()
        
    pred_df['location_price_tier'] = loc_tier
    
    # 4. Align columns precisely with the training features (X_encoded)
    # Any missing one-hot encoded columns will be automatically filled with 0
    pred_df = pred_df.reindex(columns=X_encoded.columns, fill_value=0)
    
    # 5. Impute missing values using our fitted imputer
    pred_imputed = imputer.transform(pred_df)
    
    # 6. Make prediction using our trained Gradient Boosting model
    predicted_price = gb_model.predict(pred_imputed)[0]
    
    print(f"=== Property Price Prediction ===")
    print(f"Area: {area_sqm} sq.m | Year Built: {year} | Location: {neighborhood_name}")
    print(f"Estimated Price: **{predicted_price:,.2f} EUR**")

# Test the function with an example property in Sofia (Center)
predict_property_price(area_sqm=85, year=2020, neighborhood_name='град София, Център')

In [ ]:
import joblib

joblib.dump(gb_model, 'gb_real_estate_model.pkl')
joblib.dump(imputer, 'imputer.pkl')

print("=== Model and imputer successfully saved! ===")

In [9]:
import joblib
import pandas as pd

# Зареждаме CSV файла направо тук, за да си гарантираме, че имаме данните
df = pd.read_csv('processed/cleaned_prodazhbi_05_05_2026.csv')

# Премахваме колоните, които не са ни нужни (както при обучението)
columns_to_drop = ['price']
for col in ['pricePerSqm', 'priceFormatted', 'descriptionBg', 'listingUrl']:
    if col in df.columns:
        columns_to_drop.append(col)

for col in df.columns:
    if 'url' in col.lower() or 'id' in col.lower() or 'title' in col.lower():
        if col not in columns_to_drop:
            columns_to_drop.append(col)

X = df.drop(columns=columns_to_drop)

# Изчисляваме location_means наново, за да няма празни променливи
if 'neighborhoodBg' in df.columns:
    location_col = 'neighborhoodBg'
elif 'cityBg' in df.columns:
    location_col = 'cityBg'
else:
    location_col = None

if location_col:
    location_means = df.groupby(location_col)['price'].mean().to_dict()
    X['location_price_tier'] = df[location_col].map(location_means)
    X = X.drop(columns=[location_col])

X_encoded = pd.get_dummies(X, drop_first=True)

# Записваме всичко чисто и сигурно на диска
joblib.dump(list(X_encoded.columns), 'feature_columns.pkl')
joblib.dump(location_means, 'location_means.pkl')

print("=== УСПЕХ! Всички файлове са записани перфектно! ===")

=== УСПЕХ! Всички файлове са записани перфектно! ===
